# 02 — Preprocesamiento del corpus de earnings calls

Partimos del corpus de transcripciones generado en el primer cuaderno y construimos el corpus final que consumirán los modelos de representación (TF-IDF, FinBERT, SBERT) y la validación económica. El flujo es:

1. **Exploración de n-gramas**, que motiva las decisiones de limpieza y la lista de stopwords financieras.
2. **Segmentación Presentación / Q&A** y **eliminación del Safe Harbor Statement**.
3. **Dataset final**: Eliminación de datos repetidos de `(ticker, quarter)`, filtro de validez por número de palabras y cruce con el panel **GICS** generado en el anterior código.
4. **Lematización con spaCy** y filtrado de stopwords tras lematización.

A la salida quedan definidos `earnings_calls.parquet`, `Dataset_Transcripciones_Limpio.parquet`, `Dataset_lematizado.parquet` y `universo_efectivo.csv`.

In [1]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import spacy
from nltk.util import ngrams

In [2]:
# Rutas
ruta_raw       = "./data/raw"
ruta_processed = "./data/processed"
ruta_outputs   = "./outputs"

ruta_transcripciones_raw = os.path.join(ruta_raw,       "Transcripciones_2019_2024.parquet")
ruta_sectores_gics_raw   = os.path.join(ruta_raw,       "Sectores_GICS_raw.parquet")
ruta_earnings_calls      = os.path.join(ruta_processed, "earnings_calls.parquet")
ruta_dataset_limpio      = os.path.join(ruta_processed, "Dataset_Transcripciones_Limpio.parquet")
ruta_dataset_lematizado  = os.path.join(ruta_processed, "Dataset_lematizado.parquet")
ruta_universo_efectivo   = os.path.join(ruta_processed, "universo_efectivo.csv")
ruta_ngramas_top100      = os.path.join(ruta_outputs,   "03_ngramas_top100.csv")
ruta_ngramas_top500_uni  = os.path.join(ruta_outputs,   "03_ngramas_top500_unigramas.csv")
ruta_ngramas_top500_bi   = os.path.join(ruta_outputs,   "03_ngramas_top500_bigramas.csv")

for carpeta in [ruta_processed, ruta_outputs]:
    os.makedirs(carpeta, exist_ok=True)

# Umbrales del pipeline
MIN_TRANSCRIPT_WORDS = 500    # mínimo de palabras en la presentación para considerar texto válido
N_MAX = 10                    # rango de n-gramas a explorar
TOP_K = 100                   # top-K guardado por orden n

# Stopwords financieras (se filtran post-lematización, sobre el lema canónico).
# Listado fijado tras el análisis exploratorio de n-gramas: tokens muy frecuentes
# que no aportan señal sectorial (formulismos de apertura/cierre, conectores,
# muletillas, magnitudes recurrentes).
financial_stopwords = [
    "quarter", "year", "operator", "question", "go", "ahead",
    "think", "little", "bit", "lot", "kind", "going", "sort",
    "thank", "good", "call", "million", "billion", "term",
]

#------------------------------------------------------------------------------
# Patrones regex para la segmentación Presentación / Q&A
#------------------------------------------------------------------------------
 # Diseñados a partir del análisis de n-gramas (sección 1). Capturan los
# marcadores típicos que usa el operador o el moderador para dar paso a la
# sesión de preguntas. Las cifras entre paréntesis son la frecuencia
# observada en el corpus de cada formulación.
SEPARADORES_QA = [
    # "our/your next question comes from the line of" (11K+ ocurrencias)
    r"(?:operator\s*[:\.-]?\s*)?(?:thank\s+you\s*)?(?:our|your|the)\s+(?:first|next)\s+questions?\s+(?:comes|is)\s+from\s+(?:the\s+)?lines?",

    # "operator instructions our first question comes" (2.4K+)
    r"operator\s*[:\.-]?\s*(?:(?:thank\s+you|thanks).*?)?\s*(?:instructions?|ready|open|take).{0,60}?(?:first|next)\s+questions?",

    # "turn the call over to the operator" (16K+)
    r"(?:turn|turning|hand|handing)\s+(?:the\s+)?(?:call|conference|mic|presentation|floor|program|it|session)\s+(?:back\s+)?over\s+to\s+(?:the\s+)?(?:operator|moderator|coordinator)",

    # "conduct a question and answer session" (647+)
    r"(?:conduct|begin|start|open)\s+(?:a|the\s+)?questions?\s*(?:and|&)\s*answers?\s*(?:session)?",
    r"there\s+will\s+be\s+a\s+questions?\s*(?:and|&)\s*answers?\s*(?:session)?",

    # "we'll take our next question from" (1.6K+)
    r"operator\s*[:\.-]?\s*(?:we\s+will|we'll|well)\s+take\s+(?:our|the|your)\s+(?:first|next)\s+question",

    # "please go ahead with your question"
    r"(?:please\s+)?go\s+ahead\s+(?:with\s+)?your\s+questions?",

    # "your line is now open" (con contexto de operador; observado en transcripciones)
    r"operator\s*[:\.-]?\s*.{0,80}?(?:your\s+)?lines?\s+(?:is|are)\s+(?:now\s+)?open",
    r"(?:question|analyst).{0,50}?(?:your\s+)?lines?\s+(?:is|are)\s+(?:now\s+)?open",
]

PATRON_QA = re.compile('|'.join(SEPARADORES_QA), re.IGNORECASE | re.DOTALL)

# Patrón Safe Harbor: típica introducción del Safe Harbor
# Statement al comienzo de la presentación.
PATRON_SAFE_HARBOR = re.compile(
    r"(?:turn|turning|hand|handing)\s+(?:the\s+)?(?:call|conference|presentation|time|floor|mic)"
    r"\s+(?:back\s+)?over\s+to",
    re.IGNORECASE
)

## 1. Análisis exploratorio de n-gramas

Antes de diseñar las reglas de limpieza necesitamos saber qué expresiones aparecen con más frecuencia en el corpus. En la earnings calls el operador da paso a la Q&A con frases muy parecidas ("our next question comes from the line of...", "turn the call over to the operator..."), y los presentadores arrancan con un Safe Harbor Statement legalmente obligatorio.

Guardamos los 100 N-gramas más comunes en CSV para poder citarlos en la memoria y lo usamos como base para:

- diseñar los patrones regex que segmentan presentación y Q&A (sección 2),
- construir la lista `financial_stopwords` que se filtra después de la lematización (sección 4).

In [3]:
def ngramas_comunes(textos, n=3, k=30):
    """Top-k n-gramas más frecuentes en una colección de textos.

    Pasa cada texto a minúsculas, elimina puntuación, tokeniza por espacios
    y cuenta n-gramas con Counter.
    """
    conteo_total = Counter()
    muestra = textos.dropna()
    for texto in muestra:
        texto_limpio = re.sub(r'[^\w\s]', '', str(texto).lower())
        tokens = texto_limpio.split()
        if len(tokens) >= n:
            conteo_total.update(ngrams(tokens, n))
    return conteo_total.most_common(k)


df_raw = pd.read_parquet(ruta_transcripciones_raw)
print(f"Transcripciones cargadas: {len(df_raw):,}")

# Top-100 por cada n ∈ [1, 10], persistido en CSV. En pantalla mostramos solo
# el top-15 de cada orden, para que la salida sea legible.
registros = []
for n in range(1, N_MAX + 1):
    top_ngrams = ngramas_comunes(df_raw['transcript'], n=n, k=TOP_K)
    print(f"\n{'='*60}")
    print(f"TOP {n}-GRAMAS (mostrando los 15 primeros de {TOP_K})")
    print('='*60)
    for rango, (gram, count) in enumerate(top_ngrams, start=1):
        cadena = ' '.join(gram)
        if rango <= 15:
            print(f"  [{count:>7,}]  {cadena}")
        registros.append({
            "n": n,
            "rango": rango,
            "ngrama": cadena,
            "ocurrencias": int(count),
        })

df_top = pd.DataFrame(registros, columns=["n", "rango", "ngrama", "ocurrencias"])
df_top.to_csv(ruta_ngramas_top100, index=False)
print(f"\nTop-{TOP_K} por orden n persistido en {ruta_ngramas_top100}  ({len(df_top)} registros)")

# Ampliamos a top-500 para unigramas y bigramas
TOP_K_AUDITORIA = 500
for n_ref, ruta_csv, etiqueta in [
    (1, ruta_ngramas_top500_uni, "unigramas"),
    (2, ruta_ngramas_top500_bi,  "bigramas"),
]:
    top_ref = ngramas_comunes(df_raw['transcript'], n=n_ref, k=TOP_K_AUDITORIA)
    df_ref = pd.DataFrame(
        [{"rango": r, "ngrama": " ".join(g), "ocurrencias": int(c)}
         for r, (g, c) in enumerate(top_ref, start=1)],
        columns=["rango", "ngrama", "ocurrencias"],
    )
    df_ref.to_csv(ruta_csv, index=False)
    print(f"Top-{TOP_K_AUDITORIA} {etiqueta} persistidos en {ruta_csv}")

Transcripciones cargadas: 8,939

TOP 1-GRAMAS (mostrando los 15 primeros de 100)
  [3,726,953]  the
  [2,600,215]  and
  [2,412,254]  to
  [2,141,777]  of
  [1,768,141]  in
  [1,638,148]  we
  [1,462,728]  that
  [1,346,868]  a
  [1,274,890]  our
  [893,511]  you
  [835,302]  is
  [773,297]  for
  [767,109]  i
  [759,009]  on
  [688,670]  as

TOP 2-GRAMAS (mostrando los 15 primeros de 100)
  [514,420]  in the
  [411,151]  of the
  [231,660]  i think
  [201,235]  on the
  [186,066]  for the
  [184,417]  we have
  [180,887]  of our
  [174,274]  to be
  [172,725]  to the
  [168,747]  that we
  [165,101]  we are
  [150,148]  thank you
  [138,251]  going to
  [136,399]  and the
  [132,863]  continue to

TOP 3-GRAMAS (mostrando los 15 primeros de 100)
  [ 73,923]  in terms of
  [ 72,040]  a little bit
  [ 67,872]  a lot of
  [ 51,575]  we continue to
  [ 49,148]  some of the
  [ 47,832]  question comes from
  [ 43,510]  of the year
  [ 43,322]  as well as
  [ 41,375]  next question comes
  [

## 2. Segmentación Presentación / Q&A y eliminación del Safe Harbor

Una earnings call tiene dos sesiones de naturaleza muy distinta:

- **Presentación**: discurso preparado del equipo directivo, redactado con lenguaje cuidado y enfocado en métricas y narrativa estratégica. Es la parte que mejor define la identidad sectorial.
- **Q&A**: intervenciones de analistas en formato pregunta-respuesta, con vocabulario más variado y menos estructurado.

Los modelos de representación se entrenan **únicamente sobre la presentación**, pero conviene aislar las dos sesiones para poder estudiar la presentación de forma limpia.

`PATRON_QA` reconoce el cambio hacia el operador. La función `obtener_sesion_qa` exige dos condiciones de seguridad para aceptar un punto de corte: que se encuentre más allá del 10% del documento (o de 1500 caracteres, lo que sea mayor) y que la cola Q&A resultante tenga al menos 500 caracteres. Esto evita cortar la presentación por falsos positivos en sus primeras líneas.

A continuación, `eliminar_safe_harbor` elimina el preámbulo legal del Safe Harbor Statement, buscando el patrón `turn the call (back) over to ...` en el 12% inicial de la presentación (mínimo 1500 caracteres). El Safe Harbor no tiene contenido informativo relevante y aparece de forma muy estandarizada en todas las empresas, por lo que su eliminación reduce ruido.

In [4]:
def obtener_sesion_qa(texto):
    """Separa una transcripción en presentación y sesión de Q&A.

    Devuelve la tupla (presentacion, qa, has_qa). El punto de corte debe
    estar más allá del 10% del documento (o de 1500 caracteres, lo que
    sea mayor) y la cola Q&A debe tener al menos 500 caracteres.
    """
    texto = str(texto)
    n_chars = len(texto)
    umbral_seguridad = max(1500, int(n_chars * 0.1))

    for match in PATRON_QA.finditer(texto):
        punto_corte = match.start()
        if punto_corte > umbral_seguridad:
            presentacion = texto[:punto_corte].strip()
            qa = texto[punto_corte:].strip()
            if len(qa) < 500:
                continue
            return presentacion, qa, True
    return texto, "", False


def eliminar_safe_harbor(texto_presentacion):
    """Elimina el Safe Harbor Statement del comienzo de la presentación.

    Solo busca el patrón en el 12% inicial (mínimo 1500 caracteres), para
    no comerse mucho contenido del discurso directivo.
    """
    if not isinstance(texto_presentacion, str) or len(texto_presentacion) < 1000:
        return texto_presentacion
    limite_busqueda = max(1500, int(len(texto_presentacion) * 0.12))
    zona_inicial = texto_presentacion[:limite_busqueda]
    match = PATRON_SAFE_HARBOR.search(zona_inicial)
    if match:
        return texto_presentacion[match.end():].strip()
    return texto_presentacion


df = df_raw.copy()
resultados = df['transcript'].apply(obtener_sesion_qa)
df['presentation_bruta'] = [r[0] for r in resultados]
df['qa']                 = [r[1] for r in resultados]
df['has_qa']             = [r[2] for r in resultados]

df['presentation'] = df['presentation_bruta'].apply(eliminar_safe_harbor)

# Métricas de limpieza por documento
df['ruido_eliminado_chars'] = df['presentation_bruta'].str.len() - df['presentation'].str.len()
df['pct_eliminado'] = (
    df['ruido_eliminado_chars'] / df['presentation_bruta'].str.len().replace(0, 1)
) * 100

total          = int(len(df))
exito_qa       = int(df['has_qa'].sum())
exito_sh       = int((df['ruido_eliminado_chars'] > 0).sum())
media_ruido_sh = float(df[df['ruido_eliminado_chars'] > 0]['ruido_eliminado_chars'].mean())

df_exito = df[df['has_qa']].copy()
df_exito['len_total'] = df_exito['transcript'].str.len()
df_exito['len_qa']    = df_exito['qa'].str.len()
df_exito['ratio_qa']  = df_exito['len_qa'] / df_exito['len_total']
stats_qa = df_exito['ratio_qa'].describe()
df['ratio_presentacion'] = df['presentation'].str.len() / df['transcript'].str.len()
stats_pres = df['ratio_presentacion'].describe()

print("\nResultados de la limpieza")
print("─" * 60)
print(f"Documentos procesados:           {total}")
print(f"Separación Q&A exitosa:          {exito_qa} ({exito_qa/total:.2%})")
print(f"Safe Harbor detectado y borrado: {exito_sh} ({exito_sh/total:.2%})")
print(f"Ruido legal eliminado (media):   {media_ruido_sh:.0f} caracteres / doc.")
print(f"Ratio Q&A / total — media:       {stats_qa['mean']:.2%}")
print(f"Ratio Q&A / total — mediana:     {stats_qa['50%']:.2%}")
print(f"Prepared Remarks / total — media:   {stats_pres['mean']:.2%}")
print(f"Prepared Remarks / total — mediana: {stats_pres['50%']:.2%}")

df.to_parquet(ruta_earnings_calls, index=False)
print(f"\nGuardado en {ruta_earnings_calls}")


Resultados de la limpieza
────────────────────────────────────────────────────────────
Documentos procesados:           8939
Separación Q&A exitosa:          8100 (90.61%)
Safe Harbor detectado y borrado: 7507 (83.98%)
Ruido legal eliminado (media):   604 caracteres / doc.
Ratio Q&A / total — media:       60.09%
Ratio Q&A / total — mediana:     60.78%
Prepared Remarks / total — media:   44.55%
Prepared Remarks / total — mediana: 39.63%

Guardado en ./data/processed\earnings_calls.parquet


## 3. Dataset final: deduplicación temporal, validez y cruce con GICS

El dataset para NLP se construye en tres pasos consecutivos.

**Deduplicación econométricamente correcta.** Algunas tuplas `(ticker, quarter)` aparecen con varias emisiones (la original más una o varias reemisiones corregidas). Una reemisión publicada semanas después incorpora información posterior al evento, así que la asociaríamos al trimestre equivocado si la conserváramos. Ordenamos por `earnings_date` ascendente y conservamos la emisión más temprana con `keep='first'`.

**Filtro de validez por número de palabras.** Marcamos como `texto_valido = 1` únicamente las observaciones cuya presentación supera las **500 palabras**. Los textos por debajo del umbral no proporcionan suficiente material a los modelos de representación; el campo `presentation` se pone a `""` para no utilizarse.

**Cruce con el panel GICS.** Unimos cada `(ticker, quarter)` con el sector vigente en ese trimestre según el panel construido en el cuaderno anterior (que respeta la corrección histórica de la reforma del 20-03-2023). Por seguridad deduplicamos el panel con `keep='first'` antes del merge, aunque en condiciones normales no hay duplicados en la clave.

In [5]:
df_nlp = df.copy()

# Deduplicación temporal: conservar la emisión más temprana por (ticker, quarter)
df_nlp['earnings_date'] = pd.to_datetime(df_nlp['earnings_date'])
df_nlp = df_nlp.sort_values('earnings_date')

n_antes_dedup = int(len(df_nlp))
duplicados_mask = df_nlp.duplicated(subset=['ticker', 'quarter'], keep=False)
n_implicados = int(duplicados_mask.sum())
df_nlp = df_nlp.drop_duplicates(subset=['ticker', 'quarter'], keep='first')
n_tras_dedup = int(len(df_nlp))

print("Deduplicación temporal por (ticker, quarter):")
print(f"  Filas antes:                              {n_antes_dedup}")
print(f"  Filas implicadas en duplicados:           {n_implicados}")
print(f"  Filas tras deduplicación (keep='first'):  {n_tras_dedup}")

# Identificador único de tupla
df_nlp['tupla_id'] = df_nlp['ticker'] + "_" + df_nlp['quarter'].astype(str)

# Filtro de validez (> 500 palabras en la presentación)
df_nlp['presentation'] = df_nlp['presentation'].fillna("")
df_nlp['palabras_presentacion'] = df_nlp['presentation'].str.split().str.len()
cond_valido = (df_nlp['presentation'] != "") & (
    df_nlp['palabras_presentacion'] > MIN_TRANSCRIPT_WORDS
)
df_nlp['texto_valido'] = cond_valido.astype(int)
# Para los textos no válidos vaciamos el campo
df_nlp.loc[df_nlp['texto_valido'] == 0, 'presentation'] = ""

n_validos = int(df_nlp['texto_valido'].sum())
print(f"\nFiltro de validez (> {MIN_TRANSCRIPT_WORDS} palabras):")
print(f"  Total observaciones:        {len(df_nlp)}")
print(f"  Textos válidos:             {n_validos} ({n_validos/len(df_nlp):.2%})")
print(f"  Textos no válidos:          {len(df_nlp) - n_validos}")

Deduplicación temporal por (ticker, quarter):
  Filas antes:                              8939
  Filas implicadas en duplicados:           74
  Filas tras deduplicación (keep='first'):  8902

Filtro de validez (> 500 palabras):
  Total observaciones:        8902
  Textos válidos:             8893 (99.90%)
  Textos no válidos:          9


In [6]:
# Cruce con el panel GICS generado en el cuaderno 01.
df_gics = pd.read_parquet(ruta_sectores_gics_raw)
df_gics['quarter'] = df_gics['quarter'].astype(str)
df_nlp['quarter']  = df_nlp['quarter'].astype(str)

gics_antes = int(len(df_gics))
gics_n_dup = int(df_gics.duplicated(subset=['ticker', 'quarter'], keep=False).sum())
df_gics_sub = df_gics[['ticker', 'quarter', 'gics_sector']].drop_duplicates(
    subset=['ticker', 'quarter'], keep='first'
)
print(f"Panel GICS: {gics_antes} filas, {gics_n_dup} con (ticker, quarter) duplicado "
      f"(deduplicadas con keep='first').")

df_nlp = pd.merge(df_nlp, df_gics_sub, on=['ticker', 'quarter'], how='left')
df_nlp['sector'] = df_nlp['gics_sector']
df_nlp = df_nlp.drop(columns=['gics_sector'])

n_sin_sector = int(df_nlp['sector'].isna().sum())
print(f"Observaciones sin sector tras el merge: {n_sin_sector}")

# Selección de columnas y persistencia del dataset final
columnas_finales = [
    'tupla_id', 'ticker', 'company', 'sector', 'earnings_date',
    'quarter', 'presentation', 'has_qa', 'texto_valido', 'palabras_presentacion'
]
df_nlp = df_nlp[columnas_finales]
df_nlp.to_parquet(ruta_dataset_limpio, index=False)
print(f"\nDataset final guardado en {ruta_dataset_limpio}")
print(f"  Observaciones totales:    {len(df_nlp)}")
print(f"  Textos válidos para NLP:  {int(df_nlp['texto_valido'].sum())}")

# Universo efectivo: tickers con al menos una observación válida (intersección
# entre el universo bursátil del primer cuaderno y la disponibilidad de texto válido).
df_validos = df_nlp[df_nlp['texto_valido'] == 1]
universo_efectivo = (
    df_validos
    .groupby('ticker')
    .agg(
        n_trimestres=('quarter', 'nunique'),
        primer_trimestre=('quarter', 'min'),
        ultimo_trimestre=('quarter', 'max'),
        sector=('sector', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    )
    .reset_index()
    .sort_values(['sector', 'ticker'])
)
universo_efectivo.to_csv(ruta_universo_efectivo, index=False)
print(f"Universo efectivo: {len(universo_efectivo)} tickers con texto válido → "
      f"{ruta_universo_efectivo}")

Panel GICS: 10563 filas, 0 con (ticker, quarter) duplicado (deduplicadas con keep='first').
Observaciones sin sector tras el merge: 0

Dataset final guardado en ./data/processed\Dataset_Transcripciones_Limpio.parquet
  Observaciones totales:    8902
  Textos válidos para NLP:  8893
Universo efectivo: 454 tickers con texto válido → ./data/processed\universo_efectivo.csv


## 4. Lematización con spaCy

La última etapa del preprocesado normaliza morfológicamente los textos válidos. Usamos `en_core_web_sm` con los componentes `ner` y `parser` deshabilitados, porque solo necesitamos tokenización y lematización (las componentes deshabilitadas son costosas y no aportan a nuestro objetivo).

Para cada token, conservamos su lema en minúsculas si:

1. `token.is_alpha` es verdadero (descartamos cifras y símbolos),
2. La longitud del token original es mayor que 2 (descartamos artículos cortos y abreviaturas residuales),
3. El lema no está en `stop_set`, que es la unión de las stopwords por defecto de spaCy y la lista `financial_stopwords` definida arriba.

El filtro de stopwords se aplica **sobre el lema canónico**, no sobre la forma superficial. Esto garantiza que, por ejemplo, "questions", "Question" y "questioning" se eliminen de forma consistente (todas tienen lema `question`, que está en `financial_stopwords`).

In [7]:
def limpiar_textos_spacy(lista_textos, nlp, stopwords_custom):
    """Lematiza y filtra tokens usando spaCy.

    Reglas: token alfabético, longitud > 2, lema no en stopwords.
    """
    stop_set = set(nlp.Defaults.stop_words) | set(stopwords_custom)
    textos_limpios = []
    for doc in nlp.pipe(lista_textos, batch_size=100):
        tokens = []
        for token in doc:
            if not token.is_alpha or len(token.text) <= 2:
                continue
            lema = token.lemma_.lower()
            if lema in stop_set:
                continue
            tokens.append(lema)
        textos_limpios.append(" ".join(tokens))
    return textos_limpios

nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

df_validos = df_nlp[df_nlp['texto_valido'] == 1].copy()
textos = df_validos['presentation'].tolist()

textos_lematizados = limpiar_textos_spacy(textos, nlp, financial_stopwords)
df_validos['presentation_limpia'] = textos_lematizados

df_validos.to_parquet(ruta_dataset_lematizado, index=False)
print(f"\nGuardado en {ruta_dataset_lematizado}")

# Longitud media antes y después de la lematización
n_palabras_pre  = df_validos['palabras_presentacion'].mean()
n_palabras_post = df_validos['presentation_limpia'].str.split().str.len().mean()
print(f"  Palabras / documento (pre):  {n_palabras_pre:.0f}")
print(f"  Palabras / documento (post): {n_palabras_post:.0f}")
print(f"  Reducción media:             {(1 - n_palabras_post / n_palabras_pre):.2%}")


Guardado en ./data/processed\Dataset_lematizado.parquet
  Palabras / documento (pre):  3807
  Palabras / documento (post): 1739
  Reducción media:             54.31%


## Resumen

In [8]:
print("─" * 60)
print(f"  Transcripciones de entrada:    {len(df_raw):>5d}")
print(f"  Tras deduplicación por (ticker, quarter):    {n_tras_dedup:>5d}")
print(f"  Tras filtro de texto válido (>500 palabras): {n_validos:>5d}")
print(f"  Universo efectivo (tickers con texto):       {len(universo_efectivo):>5d}")

print("\nDistribución sectorial del dataset final")
print("─" * 60)
dist_total   = df_nlp['sector'].value_counts(dropna=False)
dist_validos = df_validos['sector'].value_counts(dropna=False)
dist = pd.DataFrame({
    "total":   dist_total,
    "validos": dist_validos,
}).fillna(0).astype(int).sort_values("validos", ascending=False)
print(dist)

────────────────────────────────────────────────────────────
  Transcripciones de entrada:     8939
  Tras deduplicación por (ticker, quarter):     8902
  Tras filtro de texto válido (>500 palabras):  8893
  Universo efectivo (tickers con texto):         454

Distribución sectorial del dataset final
────────────────────────────────────────────────────────────
                        total  validos
sector                                
Information Technology   1452     1452
Industrials              1328     1328
Financials               1192     1191
Health Care              1118     1117
Consumer Discretionary    899      899
Real Estate               618      618
Consumer Staples          575      570
Utilities                 570      569
Materials                 473      473
Energy                    359      359
Communication Services    318      317
